In [7]:
import sys
import json
import uuid
from datetime import datetime, timezone

class DataElement:
    """Сущность колонки с поддержкой описания."""
    def __init__(self, field_name: str, field_type: str, description: str = ""):
        self.field_name = field_name
        self.field_type = field_type
        self.description = description

    def to_openlineage_field(self) -> dict:
        """Конвертация в формат поля OpenLineage Schema Dataset Facet."""
        return {
            "name": self.field_name,
            "type": self.field_type,
            "description": self.description
        }


class ColumnManager:
    """Менеджер для хаотичного сбора колонок и группировки по таблицам."""
    def __init__(self):
        self._tables = {}

    def add_column(self, table_name: str, element: DataElement) -> None:
        if not isinstance(element, DataElement):
            raise TypeError(f"Expected DataElement, got {type(element).__name__}")
        
        if table_name not in self._tables:
            self._tables[table_name] = []
        self._tables[table_name].append(element)

    def get_tables(self) -> dict:
        """Возвращает сырой словарь со сгруппированными объектами колонок."""
        return self._tables


class DocumentOpenLineage:
    """Сущность верхнего уровня для генерации валидного OpenLineage RunEvent."""
    def __init__(self, 
                 job_name: str = "select_ojects", 
                 job_namespace: str = "dwh_process", 
                 datasource_namespace: str = "postgresql://dwh-server:5432/production"):
        
        self.job_name = job_name
        self.job_namespace = job_namespace
        self.datasource_namespace = datasource_namespace
        
        # Константы спецификации OpenLineage
        self.producer = "https://github.com"
        self.schema_url = "https://openlineage.io/spec/1-0-5/OpenLineage.json#/definitions/RunEvent"
        self.facet_producer = "https://github.com/OpenLineage/OpenLineage/tree/1.16.0/client/python"
        self.facet_url = "https://raw.githubusercontent.com/OpenLineage/OpenLineage/main/spec/OpenLineage.json#/definitions/SchemaDatasetFacet"

    def generate_event(self, manager: ColumnManager) -> dict:
        """Трансформирует таблицы из менеджера в структуру RunEvent."""
        inputs = []
        
        # Проходим по сгруппированным таблицам
        for table_name, columns in manager.get_tables().items():
            # Формируем список полей для фасета схемы
            fields = [col.to_openlineage_field() for col in columns]
            
            # Собираем структуру датасета
            dataset = {
                "namespace": self.datasource_namespace,
                "name": table_name,
                "facets": {
                    "schema": {
                        "_producer": self.facet_producer,
                        "_schemaURL": self.facet_url,
                        "fields": fields
                    }
                },
                "inputFacets": {}
            }
            inputs.append(dataset)

        # Собираем итоговый документ
        event = {
            "schemaURL": self.schema_url,
            "eventType": "COMPLETE",
            "eventTime": datetime.now(timezone.utc).isoformat(),
            "producer": self.producer,
            "run": {
                "runId": str(uuid.uuid4()), # Генерируем уникальный runId на каждое событие
                "facets": {}
            },
            "job": {
                "namespace": self.job_namespace,
                "name": self.job_name,
                "facets": {}
            },
            "inputs": inputs,
            "outputs": []
        }
        return event

    def print_json(self, event: dict) -> None:
        """Выводит сгенерированный документ в stdout."""
        json_output = json.dumps(event, ensure_ascii=False, indent=2)
        return(json_output + "\n")

In [8]:
# 1. Создаем менеджер колонок
manager = ColumnManager()

# 2. Внешний цикл наполняет менеджер вразнобой (добавили описания колонок)
manager.add_column("users", DataElement("user_id", "INT", "Primary Key"))
manager.add_column("orders", DataElement("order_id", "BIGINT", "Order nom"))
manager.add_column("users", DataElement("email", "VARCHAR", "User Email"))
manager.add_column("orders", DataElement("amount", "DECIMAL", "Amount"))
manager.add_column("users", DataElement("created_at", "TIMESTAMP", "Created at"))

# 3. Инициализируем объект верхнего уровня
doc_generator = DocumentOpenLineage(
    job_name="select_ojects", 
    job_namespace="dwh_process",
    datasource_namespace="postgresql://dwh-server:5432/production"
)

# 4. Генерируем структуру и печатаем её
openlineage_document = doc_generator.generate_event(manager)
print(doc_generator.print_json(openlineage_document))


{
  "schemaURL": "https://openlineage.io/spec/1-0-5/OpenLineage.json#/definitions/RunEvent",
  "eventType": "COMPLETE",
  "eventTime": "2026-07-06T13:51:26.163244+00:00",
  "producer": "https://github.com",
  "run": {
    "runId": "06211c9d-ee1a-4de5-b849-26c70bb2d41e",
    "facets": {}
  },
  "job": {
    "namespace": "dwh_process",
    "name": "select_ojects",
    "facets": {}
  },
  "inputs": [
    {
      "namespace": "postgresql://dwh-server:5432/production",
      "name": "users",
      "facets": {
        "schema": {
          "_producer": "https://github.com/OpenLineage/OpenLineage/tree/1.16.0/client/python",
          "_schemaURL": "https://raw.githubusercontent.com/OpenLineage/OpenLineage/main/spec/OpenLineage.json#/definitions/SchemaDatasetFacet",
          "fields": [
            {
              "name": "user_id",
              "type": "INT",
              "description": "Primary Key"
            },
            {
              "name": "email",
              "type": "VARCHAR